# COVID-19 Research Papers Analysis

This notebook contains a comprehensive analysis of COVID-19 research papers dataset. We'll perform data exploration, cleaning, analysis, and visualization, ultimately creating a Streamlit application to showcase our findings.

## Setup and Data Loading
First, let's import the necessary libraries and load our dataset.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import streamlit as st
from wordcloud import WordCloud
from datetime import datetime

# Set style for visualizations
plt.style.use('seaborn')
sns.set_palette("husl")

# Load the dataset
df = pd.read_csv('metadata.csv')
print("Dataset loaded successfully!")

## 1. Basic Data Exploration

Let's start by examining the basic characteristics of our dataset.

In [ ]:
# Check DataFrame dimensions
print("Dataset Dimensions:")
print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")

# Display the first few rows
print("\nFirst few rows of the dataset:")
display(df.head())

### Data Types and Missing Values
Let's examine the data types of each column and check for missing values.

In [ ]:
# Display data types of each column
print("Data Types of Each Column:")
print(df.dtypes)

# Check for missing values
print("\nMissing Values in Each Column:")
missing_values = df.isnull().sum()
missing_percentages = (missing_values / len(df)) * 100
missing_info = pd.DataFrame({
    'Missing Values': missing_values,
    'Percentage Missing': missing_percentages
})
display(missing_info)

### Basic Statistics for Numerical Columns
Let's generate basic statistics for the numerical columns in our dataset.

In [ ]:
# Generate basic statistics for numerical columns
print("Basic Statistics for Numerical Columns:")
display(df.describe())

## 2. Data Cleaning and Preparation

Let's clean the dataset and prepare it for analysis. We'll handle missing values, convert date formats, and create new features.

In [ ]:
# Create a copy of the original dataset
df_cleaned = df.copy()

# Handle missing values based on column type
# For text columns, fill with "Unknown"
text_columns = df_cleaned.select_dtypes(include=['object']).columns
df_cleaned[text_columns] = df_cleaned[text_columns].fillna('Unknown')

# Convert date columns to datetime format
if 'publish_time' in df_cleaned.columns:
    df_cleaned['publish_time'] = pd.to_datetime(df_cleaned['publish_time'], errors='coerce')
    
# Extract year from publication date
df_cleaned['publication_year'] = df_cleaned['publish_time'].dt.year

# Create word count columns for title and abstract
if 'title' in df_cleaned.columns:
    df_cleaned['title_word_count'] = df_cleaned['title'].str.split().str.len()
if 'abstract' in df_cleaned.columns:
    df_cleaned['abstract_word_count'] = df_cleaned['abstract'].str.split().str.len()

# Display information about the cleaned dataset
print("Cleaned Dataset Info:")
display(df_cleaned.info())

### Verify Data Cleaning Results
Let's check the results of our data cleaning process by comparing missing values before and after cleaning.

In [ ]:
# Compare missing values before and after cleaning
missing_before = df.isnull().sum()
missing_after = df_cleaned.isnull().sum()

comparison = pd.DataFrame({
    'Missing Before': missing_before,
    'Missing After': missing_after,
    'Difference': missing_before - missing_after
})

print("Missing Values Comparison:")
display(comparison)

# Verify new features
print("\nNew Features Preview:")
new_features = ['publication_year', 'title_word_count', 'abstract_word_count']
display(df_cleaned[new_features].describe())

## 3. Data Analysis and Visualization

Now let's analyze the cleaned dataset and create visualizations to better understand the data.

In [ ]:
# Count papers by publication year
yearly_counts = df_cleaned['publication_year'].value_counts().sort_index()

plt.figure(figsize=(12, 6))
yearly_counts.plot(kind='line', marker='o')
plt.title('Number of Publications Over Time')
plt.xlabel('Year')
plt.ylabel('Number of Publications')
plt.grid(True)
plt.show()

# Print summary statistics
print("\nPublication Counts by Year:")
display(yearly_counts)

In [ ]:
# Identify top journals
if 'journal' in df_cleaned.columns:
    top_journals = df_cleaned['journal'].value_counts().head(10)

    plt.figure(figsize=(12, 6))
    top_journals.plot(kind='bar')
    plt.title('Top 10 Journals Publishing COVID-19 Research')
    plt.xlabel('Journal')
    plt.ylabel('Number of Publications')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

    print("\nTop 10 Journals by Publication Count:")
    display(top_journals)

In [ ]:
# Create word cloud of paper titles
if 'title' in df_cleaned.columns:
    # Combine all titles
    text = ' '.join(df_cleaned['title'].dropna())
    
    # Create and generate a word cloud image
    wordcloud = WordCloud(width=800, height=400, background_color='white').generate(text)
    
    # Display the word cloud
    plt.figure(figsize=(15, 8))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title('Word Cloud of Paper Titles')
    plt.show()
    
    # Get most frequent words
    from collections import Counter
    import re
    
    # Tokenize and count words
    words = re.findall(r'\w+', text.lower())
    word_freq = Counter(words)
    
    print("\nMost Common Words in Titles:")
    display(pd.DataFrame(word_freq.most_common(20), columns=['Word', 'Frequency']))

## 4. Streamlit Application

Now let's create a Streamlit application to make our analysis interactive. We'll save this code in a separate file called `app.py`.